Imports

In [ ]:
import os, vertexai, json
from pdf2image import convert_from_path
from vertexai.generative_models import GenerativeModel, Part

Parameters

In [ ]:
# Using r"..." (raw string) to avoid errors with \ on Windows
INPUT_FOLDER = r"C:\Path\To\Your\Input\Folder"
OUTPUT_FILE = r"C:\Path\To\Your\Output\results.json"

# Convert invoices to images #

In [ ]:
def convert_pdf_to_images(pdf_path, output_dir="temp_images"):
    """Converts a PDF file into a series of images (one per page)."""
    os.makedirs(output_dir, exist_ok=True)
    base_name = os.path.basename(pdf_path).rsplit(".", 1)[0]

    images = convert_from_path(pdf_path, dpi=350)
    image_paths = []

    for i, img in enumerate(images, start=1):
        img_path = os.path.join(output_dir, f"{base_name}_page{i}.png")
        img.save(img_path, "PNG")
        image_paths.append(img_path)

    return image_paths

# Clean output text  #

In [ ]:
def extract_from_text(text):
    """Cleans the text returned by the AI to keep only the pure JSON array."""
    text = text.strip()
    # Removes markdown tags ```json ... ``` if present
    if text.startswith("```json"):
        text = text[7:]
    elif text.startswith("```"):
        text = text[3:]
    
    if text.endswith("```"):
        text = text[:-3]
    return text

# Extraction prompt #

In [ ]:
EXTRACTION_PROMPT = """
You are an expert in reading multi-page invoices.
Analyze all pages of the PDF and return a strictly valid JSON LIST.
Classify by page, not by the whole PDF.
Each PDF can have multiple pages containing distinct invoices.

WARNING - FILTERING OUT IRRELEVANT PAGES:
- If you identify the "[ICON_NAME]" icon on the page: strictly classify it as "non_invoice".
- Pages that do not belong to one of the following 3 forwarders {[FORWARDER_1] ; [FORWARDER_2] ; [FORWARDER_3]} must be strictly classified as "non_invoice".

CRUCIAL WARNING - ENTITY DISTINCTION:
- "recipient": The CLIENT (Company) to whom the invoice is addressed (e.g., [COMPANY_A], [COMPANY_B]).
- "destination": The GEOGRAPHICAL LOCATION of arrival (City/Country).
  * Look for keywords: "POD", "Port of Discharge", "Destination", "Delivery Location".
  * Typical values: "[COUNTRY_1]", "[CITY_1]", "[COUNTRY_2]".
  * IT IS STRICTLY FORBIDDEN to put a company name here.

Expected JSON List format:
[
  {
    "classification": "original_invoice | non_invoice",
    "invoice_number": "<string>",
    "file_number": "<string>",
    "invoice_details": {
      "forwarder": "STRING",  // Choose from: {[FORWARDER_1] ; [FORWARDER_2] ; [FORWARDER_3]}
      "date": "STRING",
      "destination": "STRING"
    }   
  }
]

Rules:
- Always return a JSON LIST.
- Do not output any text outside of the JSON block.
- If the classification is "non_invoice", the other fields can be null.
- Extract the forwarder strictly from these 3 options: {[FORWARDER_1] ; [FORWARDER_2] ; [FORWARDER_3]}. 
- If a PDF contains multiple invoice pages, you must create a separate object for each detected invoice inside the output list.
"""

# Extraction pipeline #

In [ ]:
def extract_invoices(image_paths: list) -> dict:
    """Extracts invoice data from a list of images using Vertex AI Gemini model."""
    # Exception for empty folder
    if not image_paths:
        return []
    
    vertexai.init(
        project="your-gcp-project-id", # ANONYMIZED
        location="europe-central2",
        api_endpoint="europe-central2-aiplatform.googleapis.com"
    )
    model = GenerativeModel("gemini-2.0-flash")

    try:
        parts = []
        for path in image_paths:
            with open(path, "rb") as f:
                img_bytes = f.read()
            parts.append(Part.from_data(img_bytes, mime_type="image/png"))

        response = model.generate_content(
            [EXTRACTION_PROMPT] + parts,
            generation_config={"max_output_tokens": 8000, "temperature": 0},
            stream=False
        )

        # Parse the JSON to return a usable object
        cleaned_json = extract_from_text(response.text.strip())
        return json.loads(cleaned_json)

    except Exception as e:
        print(f" Error during invoice extraction: {e}")
        return {"error": str(e)}

# Batch processing #

In [ ]:
def process_invoices_folder():
    """Scans the folder, updates the existing JSON, and skips already processed invoices."""
    
    if not os.path.exists(INPUT_FOLDER):
        os.makedirs(INPUT_FOLDER)
        print(f"Folder '{INPUT_FOLDER}' created. Please drop your PDFs inside.")
        return

    # A. LOAD EXISTING DATA
    summary_list = []
    
    if os.path.exists(OUTPUT_FILE):
        try:
            with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
                summary_list = json.load(f)
            print(f"Existing JSON file loaded: {len(summary_list)} invoices already stored.")
        except json.JSONDecodeError:
            print("JSON file corrupted or empty. Starting from scratch.")
            summary_list = []
    
    # Keep track of already processed files
    processed_files = {entry.get("file_name") for entry in summary_list}

    files = [f for f in os.listdir(INPUT_FOLDER) if f.lower().endswith('.pdf')]
    print(f"Processing {len(files)} files found in the folder...")
    
    new_count = 0

    for filename in files:
        if filename in processed_files:
             print(f"File already processed: {filename}")
             continue 

        full_path = os.path.join(INPUT_FOLDER, filename)
        abs_path = os.path.abspath(full_path)
        print(f"Analyzing: {filename}")

        # Conversion and Extraction
        image_paths = convert_pdf_to_images(full_path)
        data = extract_invoices(image_paths)
        
        # Building the summary
        if data and isinstance(data, list) and len(data) > 0:
            found_any_new = False
            count_in_pdf = 0
            
            for invoice_info in data:
                # --- FILTER 1: Useless Pages ---
                if invoice_info.get("classification") == "non_invoice":
                    continue

                # New invoice data (Standardization)
                new_num = str(invoice_info.get("invoice_number", "")).strip().upper()
                new_file_num = str(invoice_info.get("file_number", "")).strip().upper()
                
                new_details = invoice_info.get("invoice_details", {})
                new_trans = str(new_details.get("forwarder", "")).strip().upper()
                new_dest = str(new_details.get("destination", "")).strip().upper()
                
                # --- FILTER 2: Duplicate Check (STRICT RULES) ---
                is_duplicate = False
                INVALID_VALUES = ["", "NOT FOUND", "NULL", "NONE", "UNCLASSIFIED"]

                for existing_item in summary_list:
                    existing_details = existing_item.get("invoice_details", {})
                    
                    ex_num = str(existing_item.get("invoice_number", "")).strip().upper()
                    ex_file_num = str(existing_item.get("file_number", "")).strip().upper()
                    ex_trans = str(existing_details.get("forwarder", "")).strip().upper()
                    ex_dest = str(existing_details.get("destination", "")).strip().upper()
                    
                    # BASE CONDITIONS (Common to both cases)
                    base_match = (
                        (ex_trans == new_trans) and (new_trans not in INVALID_VALUES) and
                        (ex_dest == new_dest) and (new_dest not in INVALID_VALUES)
                    )

                    if base_match:
                        # CASE 1: Same Forwarder + Same Dest + Same Invoice
                        if (ex_num == new_num) and (new_num not in INVALID_VALUES):
                            is_duplicate = True
                            print(f"Duplicate detected (Invoice already exists): {new_num}")
                            break
                        
                        # CASE 2: Same Forwarder + Same Dest + Same File Number
                        elif (ex_file_num == new_file_num) and (new_file_num not in INVALID_VALUES):
                            is_duplicate = True
                            print(f"Duplicate detected (File/Case already exists): {new_file_num}")
                            break
                
                if is_duplicate:
                    continue

                # --- REGISTRATION ---
                count_in_pdf += 1
                new_count += 1
                found_any_new = True
                
                entry = {
                    "file_name": filename,
                    "file_path": abs_path,
                    "classification": invoice_info.get("classification", "Unclassified"),
                    "invoice_number": invoice_info.get("invoice_number", "Not found"),
                    "file_number": invoice_info.get("file_number", "Not found"),
                    "invoice_details": {
                        "forwarder": new_details.get("forwarder", "Not found"),
                        "date": new_details.get("date", "Not found"),
                        "destination": new_details.get("destination", "Not found")
                    }
                }
                
                summary_list.append(entry)
                print(f"Added: {entry['invoice_details']['forwarder']} -> {entry['invoice_details']['destination']} (Invoice N° {entry['invoice_number']})")
            
            if not found_any_new and count_in_pdf == 0:
                print("No new data added for this file.")

        else:
            print("Failed to read or empty content.")

        # Cleanup temp images
        for p in image_paths:
            if os.path.exists(p): os.remove(p)
        
        # Progressive save
        with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
            json.dump(summary_list, f, indent=4, ensure_ascii=False)

    print(f"Final summary updated in: {OUTPUT_FILE}")
    print(f"{new_count} new invoices added in total.")
    
    if os.path.exists("temp_images"):
        try: os.rmdir("temp_images")
        except: pass

Main execution

In [ ]:
if __name__ == "__main__":
    process_invoices_folder()

# Observations

The previous single-LLM approach struggled with text detection issues. Therefore, we adopted a new approach: feeding the text extracted via OCR directly into the LLM. This hybrid OCR + LLM pipeline ensures much more reliable data extraction.